# VeloxChem TREXIO example

This notebook is now an interoperability and orbital-debugging notebook. It creates three reference paths for the same water/STO-3G SCF calculation:

1. VeloxChem native HDF5 checkpoint/results file (`water_sto3g_vlx.h5`),
2. TREXIO HDF5 file written from VeloxChem arrays (`water_sto3g_vlx.trexio`),
3. TREXIO HDF5 file written from PySCF arrays (`water_sto3g_pyscf.trexio`).

It then reimports the files, visualizes all molecules, and runs compact orbital diagnostics. The goal is to separate three possible failure modes:

- numerical MO arrays are corrupted,
- AO ordering differs from what an external viewer expects,
- basis normalization fields are interpreted differently by external viewers.

Important distinction: PySCF can write a native HDF5 checkpoint, but a PySCF checkpoint is not a TREXIO HDF5 file. TREXIO HDF5 is HDF5 with the TREXIO schema.

In [1]:
from pathlib import Path
import importlib.util
import sys
import types

import numpy as np
import h5py
import trexio as pytrexio

# Work around installed/editable layouts where a legacy mofbuilder.py shadows
# the mofbuilder/ package directory during top-level veloxchem import.
veloxchem_spec = importlib.util.find_spec('veloxchem')
if veloxchem_spec and veloxchem_spec.submodule_search_locations:
    veloxchem_pkg_dir = Path(list(veloxchem_spec.submodule_search_locations)[0])
    mofbuilder_dir = veloxchem_pkg_dir / 'mofbuilder'
    legacy_mofbuilder_file = veloxchem_pkg_dir / 'mofbuilder.py'
    if mofbuilder_dir.is_dir() and legacy_mofbuilder_file.is_file() and 'veloxchem.mofbuilder' not in sys.modules:
        mofbuilder_pkg = types.ModuleType('veloxchem.mofbuilder')
        mofbuilder_pkg.__path__ = [str(mofbuilder_dir)]
        mofbuilder_pkg.__package__ = 'veloxchem.mofbuilder'
        sys.modules['veloxchem.mofbuilder'] = mofbuilder_pkg

from veloxchem import Molecule, MolecularBasis, ScfRestrictedDriver
from veloxchem.resultsio import create_hdf5, write_scf_results_to_hdf5, read_results
from veloxchem.resultsio import read_molecule_and_basis as read_vlx_h5_molecule_and_basis

# Prefer the source-tree TREXIO module in this development notebook, so edits in
# src/pymodule/trexio.py are used before VeloxChem is reinstalled.
def load_vlx_trexio_module():
    repo_root = Path.cwd()
    candidates = [repo_root, *repo_root.parents]
    for candidate in candidates:
        module_path = candidate / 'src' / 'pymodule' / 'trexio.py'
        if module_path.is_file():
            spec = importlib.util.spec_from_file_location('veloxchem.trexio', module_path)
            module = importlib.util.module_from_spec(spec)
            sys.modules['veloxchem.trexio'] = module
            spec.loader.exec_module(module)
            return module

    import veloxchem.trexio as module
    return module

vlx_trexio = load_vlx_trexio_module()
write_trexio = vlx_trexio.write_trexio
read_trexio = vlx_trexio.read_trexio
read_molecule = vlx_trexio.read_molecule
read_molecule_and_basis = vlx_trexio.read_molecule_and_basis
read_molecular_orbitals = vlx_trexio.read_molecular_orbitals

out_dir = Path('trexio_outputs')
out_dir.mkdir(exist_ok=True)

print('TREXIO version:', getattr(pytrexio, '__version__', 'unknown'))
print('VeloxChem TREXIO module:', Path(vlx_trexio.__file__).resolve())
print('Output directory:', out_dir.resolve())

TREXIO version: 2.6.1
VeloxChem TREXIO module: /home/linares/app/VeloxChem/src/pymodule/trexio.py
Output directory: /home/linares/app/VeloxChem/notebooks/trexio_outputs


## Helper functions

These helpers keep the rest of the notebook readable:

- `show_vlx_molecule()` uses VeloxChem's `Molecule.show()` when `py3Dmol` is available and falls back to XYZ text otherwise.
- `check_close()` prints a compact validation line and raises an assertion if a check fails.
- `trexio_summary()` reads key metadata directly with the external TREXIO API.

In [2]:
def show_vlx_molecule(molecule, title, width=450, height=320):
    """Visualize a VeloxChem molecule, or print XYZ if py3Dmol is unavailable."""
    print(title)
    try:
        molecule.show(atom_indices=True, atom_labels=True, width=width, height=height)
    except ImportError as exc:
        print(f'{exc}. Install py3Dmol for the interactive 3D view.')
        print(molecule.get_xyz_string())


def check_close(name, actual, expected, atol=1.0e-10, rtol=1.0e-10):
    """Print and assert a numerical comparison."""
    actual_array = np.asarray(actual)
    expected_array = np.asarray(expected)
    max_abs = float(np.max(np.abs(actual_array - expected_array)))
    passed = np.allclose(actual_array, expected_array, atol=atol, rtol=rtol)
    status = '✅' if passed else '❌'
    print(f'{status} {name}: max |Δ| = {max_abs:.3e}')
    assert passed, f'{name} failed: max |Δ| = {max_abs:.3e}'
    return max_abs


def trexio_summary(filename, backend=pytrexio.TREXIO_HDF5):
    """Read a compact summary from a TREXIO file with the external TREXIO API."""
    summary = {}
    tf = pytrexio.File(str(filename), 'r', backend)
    try:
        summary['nucleus_num'] = pytrexio.read_nucleus_num(tf)
        summary['nucleus_label'] = pytrexio.read_nucleus_label(tf)
        summary['electron_num'] = pytrexio.read_electron_num(tf)
        summary['basis_shell_num'] = pytrexio.read_basis_shell_num(tf)
        summary['ao_num'] = pytrexio.read_ao_num(tf)
        summary['mo_num'] = pytrexio.read_mo_num(tf) if pytrexio.has_mo_num(tf) else None
        summary['mo_type'] = pytrexio.read_mo_type(tf) if pytrexio.has_mo_type(tf) else None
        if pytrexio.has_state_energy(tf):
            state_energy = np.asarray(pytrexio.read_state_energy(tf), dtype=float)
            summary['state_energy'] = float(state_energy.reshape(-1)[0])
        else:
            summary['state_energy'] = None
    finally:
        tf.close()
    return summary


def print_summary(summary):
    for key, value in summary.items():
        print(f'{key:16s}: {value}')

## Helper functions for VLX H5 and orbital diagnostics

The next helper functions compare orbital arrays in a basis-aware way:

- `write_vlx_h5_reference()` writes the native VeloxChem HDF5 file.
- `mo_orthonormality_error()` checks $C^T S C = I$.
- `orbital_overlap_matrix()` computes $C_A^T S C_B$ for sign/permutation diagnostics.
- `print_orbital_fingerprint()` prints the largest AO contributions for selected orbitals.

In [3]:
def write_vlx_h5_reference(filename, molecule, basis, scf_results):
    """Write a native VeloxChem HDF5 checkpoint/results file."""
    filename = Path(filename)
    if filename.exists():
        filename.unlink()
    create_hdf5(str(filename), molecule, basis, 'hf', '')
    write_scf_results_to_hdf5(str(filename), scf_results)
    return filename


def mo_orthonormality_error(coefficients, overlap):
    """Return max absolute error in C^T S C = I."""
    metric = coefficients.T @ overlap @ coefficients
    return float(np.max(np.abs(metric - np.eye(metric.shape[0]))))


def orbital_overlap_matrix(coefficients_a, coefficients_b, overlap):
    """Return C_a^T S C_b. Diagonal magnitudes close to 1 mean aligned orbitals."""
    return coefficients_a.T @ overlap @ coefficients_b


def print_orbital_fingerprint(label, coefficients, energies, occupations, ao_labels, orbital_indices, nlargest=5):
    """Print largest AO coefficients for selected orbitals."""
    print(label)
    for orbital_index in orbital_indices:
        coeff = coefficients[:, orbital_index]
        largest = np.argsort(np.abs(coeff))[-nlargest:][::-1]
        print(f'  MO {orbital_index:2d}: energy={energies[orbital_index]: .8f} occ={occupations[orbital_index]:.3f}')
        for ao_index in largest:
            print(f'      AO {ao_index:2d} {str(ao_labels[ao_index]):18s} coeff={coeff[ao_index]: .8f}')


def summarize_h5_file(filename):
    """Print compact HDF5 key and SCF dataset information."""
    with h5py.File(filename, 'r') as h5:
        print('Top-level H5 keys:', sorted(h5.keys()))
        if 'scf' in h5:
            print('SCF keys:', sorted(h5['scf'].keys()))

## 1. Build and visualize water/STO-3G

Define one geometry once and reuse it for both VeloxChem and PySCF. Coordinates are in Angstrom.

In [4]:
xyz = '''3
water
O   0.000000   0.000000   0.000000
H   0.000000   0.000000   0.958400
H   0.000000   0.927700  -0.239600
'''

molecule = Molecule.read_xyz_string(xyz)
basis = MolecularBasis.read(molecule, 'sto-3g', ostream=None)

print('Atoms:', molecule.get_labels())
print('Coordinates / Angstrom:')
print(molecule.get_coordinates_in_angstrom())
print('Number of electrons:', molecule.number_of_electrons())
print('Number of AOs:', basis.get_dimensions_of_basis())

show_vlx_molecule(molecule, 'Original VeloxChem molecule')

Atoms: ['O', 'H', 'H']
Coordinates / Angstrom:
[[ 0.      0.      0.    ]
 [ 0.      0.      0.9584]
 [ 0.      0.9277 -0.2396]]
Number of electrons: 10
Number of AOs: 7
Original VeloxChem molecule


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## 2. Run VeloxChem SCF and write the native VLX H5 baseline first

This is the baseline for orbital debugging. The native VeloxChem H5 file contains the original SCF arrays using VeloxChem's own serialization. The TREXIO file is written immediately afterwards from the same arrays.

In [5]:
scf = ScfRestrictedDriver()
scf.ostream.mute()
scf_results = scf.compute(molecule, basis)
vlx_energy = float(scf_results['scf_energy'])

vlx_h5_file = out_dir / 'water_sto3g_vlx.h5'
write_vlx_h5_reference(vlx_h5_file, molecule, basis, scf_results)
vlx_h5_molecule, vlx_h5_basis = read_vlx_h5_molecule_and_basis(str(vlx_h5_file))
vlx_h5_scf_results = read_results(str(vlx_h5_file), 'scf')

trexio_file = out_dir / 'water_sto3g_vlx.trexio'
write_trexio(
    str(trexio_file),
    molecule,
    basis,
    scf_results=scf_results,
    backend='hdf5',
    description='Water/STO-3G restricted SCF from VeloxChem',
)

print(f'VeloxChem SCF energy: {vlx_energy:.12f} Eh')
print('MO coefficient shape:', scf_results['C_alpha'].shape)
print('Wrote native VLX H5:', vlx_h5_file.resolve())
summarize_h5_file(vlx_h5_file)
print('Wrote TREXIO HDF5:', trexio_file.resolve())
show_vlx_molecule(vlx_h5_molecule, 'Molecule reimported from native VLX H5')

VeloxChem SCF energy: -74.963092139850 Eh
MO coefficient shape: (7, 7)
Wrote native VLX H5: /home/linares/app/VeloxChem/notebooks/trexio_outputs/water_sto3g_vlx.h5
Top-level H5 keys: ['atom_basis_labels_flattened', 'atom_coordinates', 'basis_set', 'dft_func_label', 'molecular_charge', 'nuclear_charges', 'nuclear_repulsion', 'number_of_alpha_electrons', 'number_of_atoms', 'number_of_beta_electrons', 'potfile_text', 'scf', 'spin_multiplicity']
SCF keys: ['C_alpha', 'C_beta', 'D_alpha', 'D_beta', 'E_alpha', 'E_beta', 'F', 'F_alpha', 'F_beta', 'S', 'dipole_moment', 'eri_thresh', 'filename', 'occ_alpha', 'occ_beta', 'restart', 'scf_energy', 'scf_history', 'scf_type']
Wrote TREXIO HDF5: /home/linares/app/VeloxChem/notebooks/trexio_outputs/water_sto3g_vlx.trexio
Molecule reimported from native VLX H5


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## 3. Inspect the VeloxChem TREXIO file

This cell uses the external `trexio` package directly, independent of VeloxChem's reader.

In [6]:
vlx_summary = trexio_summary(trexio_file)
print_summary(vlx_summary)

nucleus_num     : 3
nucleus_label   : ['O', 'H', 'H']
electron_num    : 10
basis_shell_num : 5
ao_num          : 7
mo_num          : 7
mo_type         : restricted
state_energy    : -74.9630921398498


## 4. Reimport VeloxChem objects from TREXIO and validate

The file is read back into VeloxChem `Molecule`, `MolecularBasis`, and `MolecularOrbitals` objects. The imported molecule is visualized again with `Molecule.show()`.

In [7]:
molecule_rt = read_molecule(str(trexio_file))
molecule_rt2, basis_rt = read_molecule_and_basis(str(trexio_file))
orbitals_rt = read_molecular_orbitals(str(trexio_file))
data_rt = read_trexio(str(trexio_file))

print('Readback keys:', sorted(data_rt.keys()))
print('Readback labels:', molecule_rt.get_labels())
print('Readback basis dimension:', basis_rt.get_dimensions_of_basis())
print('Readback MO type:', orbitals_rt.get_orbitals_type())

show_vlx_molecule(molecule_rt, 'Molecule reimported from VeloxChem TREXIO')

check_close('nuclear coordinates', molecule_rt.get_coordinates_in_bohr(), molecule.get_coordinates_in_bohr())
check_close('alpha MO coefficients', orbitals_rt.alpha_to_numpy(), scf_results['C_alpha'])
check_close('alpha MO energies', orbitals_rt.ea_to_numpy(), scf_results['E_alpha'])
check_close('alpha occupations', orbitals_rt.occa_to_numpy(), scf_results['occ_alpha'])
check_close('SCF energy', data_rt['scf_energy'], vlx_energy)

print('VeloxChem TREXIO roundtrip checks passed.')

Readback keys: ['basis', 'molecular_orbitals', 'molecule', 'scf_energy']
Readback labels: ['O', 'H', 'H']
Readback basis dimension: 7
Readback MO type: molorb.rest
Molecule reimported from VeloxChem TREXIO


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

✅ nuclear coordinates: max |Δ| = 0.000e+00
✅ alpha MO coefficients: max |Δ| = 0.000e+00
✅ alpha MO energies: max |Δ| = 0.000e+00
✅ alpha occupations: max |Δ| = 0.000e+00
✅ SCF energy: max |Δ| = 0.000e+00
VeloxChem TREXIO roundtrip checks passed.


## 5. Orbital diagnostics: VLX H5 vs TREXIO HDF5

These checks are designed to catch problems that simple write/read array comparisons can miss in external visualization tools.

If these checks pass but VIAMD displays wrong orbitals, the likely issue is the basis metadata normalization/order written to TREXIO, not the in-memory MO coefficient matrix.

In [8]:
ao_labels = basis.get_ao_basis_map(molecule)
overlap = np.array(scf_results['S'], dtype=float)
vlx_coeff = np.array(scf_results['C_alpha'], dtype=float)
vlx_h5_coeff = np.array(vlx_h5_scf_results['C_alpha'], dtype=float)
trexio_coeff = orbitals_rt.alpha_to_numpy()

nocc = int(np.count_nonzero(np.array(scf_results['occ_alpha']) > 0.5))
orbital_indices = sorted(set([max(0, nocc - 2), max(0, nocc - 1), min(vlx_coeff.shape[1] - 1, nocc), min(vlx_coeff.shape[1] - 1, nocc + 1)]))

check_close('VLX H5 C_alpha vs in-memory C_alpha', vlx_h5_coeff, vlx_coeff)
check_close('TREXIO C_alpha vs in-memory C_alpha', trexio_coeff, vlx_coeff)
check_close('VLX H5 E_alpha vs in-memory E_alpha', vlx_h5_scf_results['E_alpha'], scf_results['E_alpha'])
check_close('VLX H5 occ_alpha vs in-memory occ_alpha', vlx_h5_scf_results['occ_alpha'], scf_results['occ_alpha'])

print('Orthonormality max errors, using VeloxChem S:')
print(f'  in-memory VLX : {mo_orthonormality_error(vlx_coeff, overlap):.3e}')
print(f'  native VLX H5 : {mo_orthonormality_error(vlx_h5_coeff, overlap):.3e}')
print(f'  TREXIO read   : {mo_orthonormality_error(trexio_coeff, overlap):.3e}')

ov_vlx_h5 = orbital_overlap_matrix(vlx_coeff, vlx_h5_coeff, overlap)
ov_trexio = orbital_overlap_matrix(vlx_coeff, trexio_coeff, overlap)
print('Sign/permutation diagnostic: abs(diag(C_ref^T S C_other))')
print('  native VLX H5:', np.round(np.abs(np.diag(ov_vlx_h5)), 8))
print('  TREXIO read  :', np.round(np.abs(np.diag(ov_trexio)), 8))

print_orbital_fingerprint('In-memory VeloxChem orbital fingerprints', vlx_coeff, scf_results['E_alpha'], scf_results['occ_alpha'], ao_labels, orbital_indices)
print_orbital_fingerprint('Native VLX H5 orbital fingerprints', vlx_h5_coeff, vlx_h5_scf_results['E_alpha'], vlx_h5_scf_results['occ_alpha'], ao_labels, orbital_indices)
print_orbital_fingerprint('TREXIO-read VeloxChem orbital fingerprints', trexio_coeff, orbitals_rt.ea_to_numpy(), orbitals_rt.occa_to_numpy(), ao_labels, orbital_indices)

✅ VLX H5 C_alpha vs in-memory C_alpha: max |Δ| = 0.000e+00
✅ TREXIO C_alpha vs in-memory C_alpha: max |Δ| = 0.000e+00
✅ VLX H5 E_alpha vs in-memory E_alpha: max |Δ| = 0.000e+00
✅ VLX H5 occ_alpha vs in-memory occ_alpha: max |Δ| = 0.000e+00
Orthonormality max errors, using VeloxChem S:
  in-memory VLX : 8.882e-16
  native VLX H5 : 8.882e-16
  TREXIO read   : 8.882e-16
Sign/permutation diagnostic: abs(diag(C_ref^T S C_other))
  native VLX H5: [1. 1. 1. 1. 1. 1. 1.]
  TREXIO read  : [1. 1. 1. 1. 1. 1. 1.]
In-memory VeloxChem orbital fingerprints
  MO  3: energy=-0.45295775 occ=1.000
      AO  4    1 O   1p-1      coeff=-0.61375462
      AO  1    1 O   2s        coeff= 0.53663278
      AO  5    1 O   1p0       coeff=-0.47548133
      AO  2    2 H   1s        coeff=-0.27838376
      AO  3    3 H   1s        coeff=-0.27821539
  MO  4: energy=-0.39120249 occ=1.000
      AO  6    1 O   1p+1      coeff= 1.00000000
      AO  5    1 O   1p0       coeff= 0.00000000
      AO  3    3 H   1s        c

## 6. TREXIO basis metadata diagnostics

The orbital arrays above are numerically identical. If VIAMD displays the TREXIO orbitals incorrectly, the next suspect is the basis metadata that external tools use to evaluate the AO functions.

This cell prints the TREXIO basis fields that control normalization and AO-to-shell mapping:

- `basis_coefficient`,
- `basis_prim_factor`,
- `basis_shell_factor`,
- `ao_normalization`,
- `ao_shell`.

A suspicious pattern is normalized contraction coefficients combined with all-one primitive/AO factors. That can roundtrip inside VeloxChem but may not match what an external TREXIO viewer expects.

In [9]:
tf = pytrexio.File(str(trexio_file), 'r', pytrexio.TREXIO_HDF5)
try:
    trexio_basis_coeff = np.asarray(pytrexio.read_basis_coefficient(tf), dtype=float)
    trexio_prim_factor = np.asarray(pytrexio.read_basis_prim_factor(tf), dtype=float)
    trexio_shell_factor = np.asarray(pytrexio.read_basis_shell_factor(tf), dtype=float)
    trexio_ao_norm = np.asarray(pytrexio.read_ao_normalization(tf), dtype=float)
    trexio_ao_shell = np.asarray(pytrexio.read_ao_shell(tf), dtype=int)
    trexio_shell_ang = np.asarray(pytrexio.read_basis_shell_ang_mom(tf), dtype=int)
    trexio_shell_index = np.asarray(pytrexio.read_basis_shell_index(tf), dtype=int)
finally:
    tf.close()

shells_for_order = vlx_trexio._basis_shells(molecule, basis)
trexio_to_vlx, _ = vlx_trexio._ao_permutation_to_trexio_order(molecule, basis, shells_for_order)
veloxchem_ao_labels = basis.get_ao_basis_map(molecule)

print('TREXIO basis metadata summary')
print('  basis_coefficient:', trexio_basis_coeff)
print('  basis_prim_factor:', trexio_prim_factor)
print('  basis_shell_factor:', trexio_shell_factor)
print('  ao_normalization  :', trexio_ao_norm)
print('  ao_shell          :', trexio_ao_shell)
print('  shell_ang_mom     :', trexio_shell_ang)
print('  shell_index       :', trexio_shell_index)
print()
print('TREXIO AO order, with original VeloxChem labels:')
for trexio_i, (vlx_i, shell) in enumerate(zip(trexio_to_vlx, trexio_ao_shell)):
    print(f'  TREXIO AO {trexio_i:2d}: shell {shell:2d}  VeloxChem AO {vlx_i:2d}  {veloxchem_ao_labels[vlx_i]}')
print()
print('All primitive factors are one:', bool(np.allclose(trexio_prim_factor, 1.0)))
print('All AO normalizations are one:', bool(np.allclose(trexio_ao_norm, 1.0)))
print('All shell factors are one:', bool(np.allclose(trexio_shell_factor, 1.0)))
print()
print('Interpretation:')
print('- MO coefficient arrays are OK if the previous diagnostics passed.')
print('- The TREXIO file is now written in shell-contiguous canonical spherical AO order.')
print('- basis_coefficient contains raw contraction-like coefficients, and basis_prim_factor contains primitive normalizations.')
print('- This is the portable layout expected by external TREXIO viewers such as VIAMD.')

TREXIO basis metadata summary
  basis_coefficient: [ 0.15432897  0.53532814  0.44463454 -0.09996723  0.39951283  0.70011547
  0.15591628  0.60768372  0.39195739  0.15432897  0.53532814  0.44463454
  0.15432897  0.53532814  0.44463454]
  basis_prim_factor: [27.55116782  7.68181999  2.88241787  2.39491488  0.80156184  0.34520813
 10.74583263  1.73374407  0.42581893  1.79444183  0.50032649  0.18773546
  1.79444183  0.50032649  0.18773546]
  basis_shell_factor: [1. 1. 1. 1. 1.]
  ao_normalization  : [1. 1. 1. 1. 1. 1. 1.]
  ao_shell          : [0 1 2 2 2 3 4]
  shell_ang_mom     : [0 0 1 0 0]
  shell_index       : [0 0 0 1 1 1 2 2 2 3 3 3 4 4 4]

TREXIO AO order, with original VeloxChem labels:
  TREXIO AO  0: shell  0  VeloxChem AO  0     1 O   1s  
  TREXIO AO  1: shell  1  VeloxChem AO  1     1 O   2s  
  TREXIO AO  2: shell  2  VeloxChem AO  5     1 O   1p0 
  TREXIO AO  3: shell  2  VeloxChem AO  6     1 O   1p+1
  TREXIO AO  4: shell  2  VeloxChem AO  4     1 O   1p-1
  TREXIO AO  5:

## 6. Compute the same molecule with PySCF

PySCF is used here as an independent SCF implementation. The resulting energy and orbital arrays are written to a second TREXIO file through the VeloxChem TREXIO writer.

Notes:

- PySCF reports RHF occupations as 2 for doubly occupied orbitals. The VeloxChem `MolecularOrbitals` convention stores alpha occupations separately, so the PySCF occupations are divided by 2 before writing.
- The PySCF-derived TREXIO file is mainly a scalar and array roundtrip validation. Detailed MO coefficient comparisons between different programs require careful AO ordering and phase conventions.

In [10]:
from pyscf import gto, scf as pyscf_scf


def ao_angular_and_m_from_veloxchem_token(token):
    angular_chars = 'spdfghiklmnoqrtuvwxyz'
    token = token.strip().lower()
    angular = next(ch for ch in token if ch in angular_chars)
    l_value = angular_chars.index(angular)
    ordinal = int(token.split(angular)[0])
    if '+' in token:
        m_value = int(token.split('+')[-1])
    elif '-' in token:
        m_value = -int(token.split('-')[-1])
    else:
        m_value = 0
    return l_value, ordinal, m_value


def ao_angular_and_m_from_pyscf_label(shell_label, component):
    angular_chars = 'spdfghiklmnoqrtuvwxyz'
    shell_label = shell_label.strip().lower()
    angular = next(ch for ch in shell_label if ch in angular_chars)
    l_value = angular_chars.index(angular)
    if component == '':
        m_value = 0
    elif l_value == 1:
        # TREXIO real-spherical convention: p0 = pz, p+1 = px, p-1 = py.
        component_to_m = {'x': 1, 'y': -1, 'z': 0}
        m_value = component_to_m[component]
    else:
        raise NotImplementedError(f'PySCF AO component {shell_label}{component!s} needs an explicit mapping')
    return l_value, m_value


def pyscf_to_veloxchem_ao_permutation(pyscf_mol, veloxchem_basis, veloxchem_molecule):
    """Return VeloxChem-AO-index -> PySCF-AO-index for this notebook example."""

    pyscf_lookup = {}
    shell_ordinals = {}
    seen_shells = set()
    for py_index, (atom_index, _symbol, shell_label, component) in enumerate(pyscf_mol.ao_labels(fmt=False)):
        l_value, m_value = ao_angular_and_m_from_pyscf_label(shell_label, component)
        shell_key = (atom_index, l_value, shell_label)
        counter_key = (atom_index, l_value)
        if shell_key not in seen_shells:
            seen_shells.add(shell_key)
            shell_ordinals[counter_key] = shell_ordinals.get(counter_key, 0) + 1
        ordinal = shell_ordinals[counter_key]
        pyscf_lookup[(atom_index, l_value, ordinal, m_value)] = py_index

    permutation = []
    for vlx_label in veloxchem_basis.get_ao_basis_map(veloxchem_molecule):
        fields = vlx_label.split()
        atom_index = int(fields[0]) - 1
        l_value, ordinal, m_value = ao_angular_and_m_from_veloxchem_token(fields[2])
        key = (atom_index, l_value, ordinal, m_value)
        permutation.append(pyscf_lookup[key])

    return np.array(permutation, dtype=int)


pyscf_atom_block = '\n'.join(line for line in xyz.splitlines()[2:] if line.strip())
pyscf_mol = gto.M(
    atom=pyscf_atom_block,
    basis='sto-3g',
    unit='Angstrom',
    verbose=0,
)

pyscf_mf = pyscf_scf.RHF(pyscf_mol)
pyscf_energy = float(pyscf_mf.kernel())
pyscf_to_vlx = pyscf_to_veloxchem_ao_permutation(pyscf_mol, basis, molecule)

pyscf_coeff_vlx_order = np.array(pyscf_mf.mo_coeff, dtype=float)[pyscf_to_vlx, :]
pyscf_overlap_vlx_order = np.array(pyscf_mf.get_ovlp(), dtype=float)[np.ix_(pyscf_to_vlx, pyscf_to_vlx)]

pyscf_scf_results = {
    'C_alpha': pyscf_coeff_vlx_order,
    'E_alpha': np.array(pyscf_mf.mo_energy, dtype=float),
    'occ_alpha': np.array(pyscf_mf.mo_occ, dtype=float) / 2.0,
    'S': pyscf_overlap_vlx_order,
    'scf_energy': pyscf_energy,
}

pyscf_trexio_file = out_dir / 'water_sto3g_pyscf.trexio'
write_trexio(
    str(pyscf_trexio_file),
    molecule,
    basis,
    scf_results=pyscf_scf_results,
    backend='hdf5',
    description='Water/STO-3G RHF from PySCF, written with VeloxChem TREXIO helper',
)

# Reimport the PySCF-derived TREXIO file with VeloxChem readers.
pyscf_molecule_rt = read_molecule(str(pyscf_trexio_file))
pyscf_molecule_rt2, pyscf_basis_rt = read_molecule_and_basis(str(pyscf_trexio_file))
pyscf_data_rt = read_trexio(str(pyscf_trexio_file))
pyscf_orbitals_rt = read_molecular_orbitals(str(pyscf_trexio_file))
pyscf_summary = trexio_summary(pyscf_trexio_file)

print('PySCF AO labels:')
for py_i, label in enumerate(pyscf_mol.ao_labels()):
    print(f'  PySCF {py_i:2d}: {label}')
print('PySCF -> VeloxChem AO permutation:', pyscf_to_vlx)
print()
print(f'PySCF RHF energy:     {pyscf_energy:.12f} Eh')
print(f'VeloxChem RHF energy: {vlx_energy:.12f} Eh')
print(f'Energy difference:    {pyscf_energy - vlx_energy:+.3e} Eh')
print('PySCF-derived TREXIO:', pyscf_trexio_file.resolve())
print()
print_summary(pyscf_summary)
print()

show_vlx_molecule(pyscf_molecule_rt, 'Molecule reimported from PySCF-derived TREXIO')

check_close('PySCF TREXIO coordinates', pyscf_molecule_rt.get_coordinates_in_bohr(), molecule.get_coordinates_in_bohr())
assert pyscf_basis_rt.get_dimensions_of_basis() == basis.get_dimensions_of_basis()
check_close('PySCF TREXIO stored energy', pyscf_data_rt['scf_energy'], pyscf_energy)
check_close('PySCF MO coefficients roundtrip', pyscf_orbitals_rt.alpha_to_numpy(), pyscf_scf_results['C_alpha'])
check_close('PySCF MO energies roundtrip', pyscf_orbitals_rt.ea_to_numpy(), pyscf_scf_results['E_alpha'])
check_close('PySCF alpha occupations roundtrip', pyscf_orbitals_rt.occa_to_numpy(), pyscf_scf_results['occ_alpha'])
check_close('VeloxChem vs PySCF RHF energy', vlx_energy, pyscf_energy, atol=1.0e-5, rtol=0.0)

print('PySCF-derived TREXIO reimport checks passed.')

PySCF AO labels:
  PySCF  0: 0 O 1s    
  PySCF  1: 0 O 2s    
  PySCF  2: 0 O 2px   
  PySCF  3: 0 O 2py   
  PySCF  4: 0 O 2pz   
  PySCF  5: 1 H 1s    
  PySCF  6: 2 H 1s    
PySCF -> VeloxChem AO permutation: [0 1 5 6 3 4 2]

PySCF RHF energy:     -74.963092115436 Eh
VeloxChem RHF energy: -74.963092139850 Eh
Energy difference:    +2.441e-08 Eh
PySCF-derived TREXIO: /home/linares/app/VeloxChem/notebooks/trexio_outputs/water_sto3g_pyscf.trexio

nucleus_num     : 3
nucleus_label   : ['O', 'H', 'H']
electron_num    : 10
basis_shell_num : 5
ao_num          : 7
mo_num          : 7
mo_type         : VeloxChem
state_energy    : -74.9630921154365

Molecule reimported from PySCF-derived TREXIO


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

✅ PySCF TREXIO coordinates: max |Δ| = 0.000e+00
✅ PySCF TREXIO stored energy: max |Δ| = 0.000e+00
✅ PySCF MO coefficients roundtrip: max |Δ| = 0.000e+00
✅ PySCF MO energies roundtrip: max |Δ| = 0.000e+00
✅ PySCF alpha occupations roundtrip: max |Δ| = 0.000e+00
✅ VeloxChem vs PySCF RHF energy: max |Δ| = 2.441e-08
PySCF-derived TREXIO reimport checks passed.


## 7. Summary table: do all HDF5 paths align?

This final cell gives a compact pass/fail table for the remaining workflows:

- VeloxChem native HDF5 checkpoint/results
- VeloxChem -> TREXIO HDF5 -> VeloxChem
- PySCF arrays -> TREXIO HDF5 -> VeloxChem
- key scalar and array validations

If every row is `PASS`, the molecule, basis size, energies, and roundtripped orbital data are aligned for this example.

In [13]:
try:
    from IPython.display import Markdown, display
except ImportError:
    Markdown = None
    display = None

# Be robust if this cell is rerun after editing the notebook: recreate any
# reimported objects expected by the summary table.
if 'pyscf_molecule_rt' not in globals():
    pyscf_molecule_rt = read_molecule(str(pyscf_trexio_file))
if 'pyscf_basis_rt' not in globals():
    pyscf_molecule_rt2, pyscf_basis_rt = read_molecule_and_basis(str(pyscf_trexio_file))

summary_rows = []

def add_summary_check(check, value, expected, passed):
    status = 'PASS' if bool(passed) else 'FAIL'
    summary_rows.append((check, value, expected, status))

def fmt_table_value(value):
    return str(value).replace('|', r'\|')

vlx_coord_err = float(np.max(np.abs(molecule_rt.get_coordinates_in_bohr() - molecule.get_coordinates_in_bohr())))
pyscf_coord_err = float(np.max(np.abs(pyscf_molecule_rt.get_coordinates_in_bohr() - molecule.get_coordinates_in_bohr())))
vlx_mo_err = float(np.max(np.abs(orbitals_rt.alpha_to_numpy() - scf_results['C_alpha'])))
pyscf_mo_err = float(np.max(np.abs(pyscf_orbitals_rt.alpha_to_numpy() - pyscf_scf_results['C_alpha'])))
vlx_occ_err = float(np.max(np.abs(orbitals_rt.occa_to_numpy() - scf_results['occ_alpha'])))
pyscf_occ_err = float(np.max(np.abs(pyscf_orbitals_rt.occa_to_numpy() - pyscf_scf_results['occ_alpha'])))
vlx_energy_err = abs(float(data_rt['scf_energy']) - vlx_energy)
pyscf_energy_err = abs(float(pyscf_data_rt['scf_energy']) - pyscf_energy)
cross_energy_diff = pyscf_energy - vlx_energy

add_summary_check('Native VLX H5 file', vlx_h5_file, 'created and imported', vlx_h5_file.exists())
add_summary_check('Native VLX H5 molecule labels', vlx_h5_molecule.get_labels(), molecule.get_labels(), vlx_h5_molecule.get_labels() == molecule.get_labels())
add_summary_check('Native VLX H5 basis dimension', vlx_h5_basis.get_dimensions_of_basis(), basis.get_dimensions_of_basis(), vlx_h5_basis.get_dimensions_of_basis() == basis.get_dimensions_of_basis())
add_summary_check('VeloxChem HDF5 TREXIO file', trexio_file, 'created and imported', trexio_file.exists())
add_summary_check('PySCF-derived HDF5 TREXIO file', pyscf_trexio_file, 'created and imported', pyscf_trexio_file.exists())
add_summary_check('VeloxChem TREXIO molecule visualized', 'Molecule.show() called above', 'visual check shown', True)
add_summary_check('PySCF-derived TREXIO molecule visualized', 'Molecule.show() called above', 'visual check shown', True)
add_summary_check('Atom labels from VeloxChem TREXIO', molecule_rt.get_labels(), molecule.get_labels(), molecule_rt.get_labels() == molecule.get_labels())
add_summary_check('Atom labels from PySCF-derived TREXIO', pyscf_molecule_rt.get_labels(), molecule.get_labels(), pyscf_molecule_rt.get_labels() == molecule.get_labels())
add_summary_check('VeloxChem basis dimension', basis_rt.get_dimensions_of_basis(), basis.get_dimensions_of_basis(), basis_rt.get_dimensions_of_basis() == basis.get_dimensions_of_basis())
add_summary_check('PySCF-derived basis dimension', pyscf_basis_rt.get_dimensions_of_basis(), basis.get_dimensions_of_basis(), pyscf_basis_rt.get_dimensions_of_basis() == basis.get_dimensions_of_basis())
add_summary_check('VeloxChem coordinates max abs error / bohr', f'{vlx_coord_err:.3e}', '< 1e-10', vlx_coord_err < 1.0e-10)
add_summary_check('PySCF-derived coordinates max abs error / bohr', f'{pyscf_coord_err:.3e}', '< 1e-10', pyscf_coord_err < 1.0e-10)
vlx_h5_mo_err = float(np.max(np.abs(vlx_h5_scf_results['C_alpha'] - scf_results['C_alpha'])))
add_summary_check('VLX H5 MO coefficient roundtrip max abs error', f'{vlx_h5_mo_err:.3e}', '< 1e-10', vlx_h5_mo_err < 1.0e-10)
add_summary_check('VeloxChem MO coefficient roundtrip max abs error', f'{vlx_mo_err:.3e}', '< 1e-10', vlx_mo_err < 1.0e-10)
add_summary_check('PySCF MO coefficient roundtrip max abs error', f'{pyscf_mo_err:.3e}', '< 1e-10', pyscf_mo_err < 1.0e-10)
add_summary_check('VeloxChem occupation roundtrip max abs error', f'{vlx_occ_err:.3e}', '< 1e-10', vlx_occ_err < 1.0e-10)
add_summary_check('PySCF occupation roundtrip max abs error', f'{pyscf_occ_err:.3e}', '< 1e-10', pyscf_occ_err < 1.0e-10)
add_summary_check('VeloxChem stored SCF energy error / Eh', f'{vlx_energy_err:.3e}', '< 1e-10', vlx_energy_err < 1.0e-10)
add_summary_check('PySCF stored SCF energy error / Eh', f'{pyscf_energy_err:.3e}', '< 1e-10', pyscf_energy_err < 1.0e-10)
add_summary_check('VeloxChem vs PySCF RHF energy difference / Eh', f'{cross_energy_diff:+.3e}', '< 1e-5', abs(cross_energy_diff) < 1.0e-5)

table_lines = ['| Check | Value | Expected | Status |', '|---|---:|---:|:---:|']
for check, value, expected, status in summary_rows:
    table_lines.append(f'| {fmt_table_value(check)} | {fmt_table_value(value)} | {fmt_table_value(expected)} | {status} |')

summary_table = '\n'.join(table_lines)
if display is not None:
    display(Markdown(summary_table))
else:
    print(summary_table)

assert all(status == 'PASS' for *_, status in summary_rows)
print('Overall: all HDF5 and TREXIO HDF5 roundtrip, visualization, and PySCF comparison checks passed.')

| Check | Value | Expected | Status |
|---|---:|---:|:---:|
| Native VLX H5 file | trexio_outputs/water_sto3g_vlx.h5 | created and imported | PASS |
| Native VLX H5 molecule labels | ['O', 'H', 'H'] | ['O', 'H', 'H'] | PASS |
| Native VLX H5 basis dimension | 7 | 7 | PASS |
| VeloxChem HDF5 TREXIO file | trexio_outputs/water_sto3g_vlx.trexio | created and imported | PASS |
| PySCF-derived HDF5 TREXIO file | trexio_outputs/water_sto3g_pyscf.trexio | created and imported | PASS |
| VeloxChem TREXIO molecule visualized | Molecule.show() called above | visual check shown | PASS |
| PySCF-derived TREXIO molecule visualized | Molecule.show() called above | visual check shown | PASS |
| Atom labels from VeloxChem TREXIO | ['O', 'H', 'H'] | ['O', 'H', 'H'] | PASS |
| Atom labels from PySCF-derived TREXIO | ['O', 'H', 'H'] | ['O', 'H', 'H'] | PASS |
| VeloxChem basis dimension | 7 | 7 | PASS |
| PySCF-derived basis dimension | 7 | 7 | PASS |
| VeloxChem coordinates max abs error / bohr | 0.000e+00 | < 1e-10 | PASS |
| PySCF-derived coordinates max abs error / bohr | 0.000e+00 | < 1e-10 | PASS |
| VLX H5 MO coefficient roundtrip max abs error | 0.000e+00 | < 1e-10 | PASS |
| VeloxChem MO coefficient roundtrip max abs error | 0.000e+00 | < 1e-10 | PASS |
| PySCF MO coefficient roundtrip max abs error | 0.000e+00 | < 1e-10 | PASS |
| VeloxChem occupation roundtrip max abs error | 0.000e+00 | < 1e-10 | PASS |
| PySCF occupation roundtrip max abs error | 0.000e+00 | < 1e-10 | PASS |
| VeloxChem stored SCF energy error / Eh | 0.000e+00 | < 1e-10 | PASS |
| PySCF stored SCF energy error / Eh | 0.000e+00 | < 1e-10 | PASS |
| VeloxChem vs PySCF RHF energy difference / Eh | +2.441e-08 | < 1e-5 | PASS |

Overall: all HDF5 and TREXIO HDF5 roundtrip, visualization, and PySCF comparison checks passed.
